# Fil Rouge RNASeq - Partie II : Manipulation de Dataframes en R

**Date :** mardi 8 octobre 2025  
**Cours :** M1 MEG - UE5 Analyse de données  
**Objectif :** Apprendre à manipuler des dataframes en R avec des données de comptage RNASeq

<div class="alert alert-info">
<b>Environnement requis :</b><br>
- Serveur Adenine (adenine.plasmabio.org)<br>
- 5 CPUs par étudiant<br>
- 10 GB RAM par étudiant<br>
- R version 4.x avec tidyverse
</div>


Dans ce TP, vous allez travailler avec des données de RNASeq de souris (Blumenthal et al., 2014) portant sur une région du chromosome 7 impliquée dans les troubles du spectre autistique. Les souris présentent soit :
- Une **délétion** de cette région
- Une **duplication** de cette région  
- Un génotype **wild type** (témoin)

Vous disposez de deux jeux de données de comptage :
1. **Vos comptages** : générés par votre pipeline STAR + featureCounts (TP6)
2. **Comptages des auteurs** : générés par leur pipeline TopHat2 + BEDTools (2013)

L'objectif est de fusionner ces deux jeux de données pour comparer les résultats des deux approches bioinformatiques.

## Objectifs d'apprentissage

À la fin de ce TP, vous saurez :
1. **Lire des fichiers** de comptage dans R en gérant les en-têtes
2. **Manipuler des dataframes** : renommer, réordonner, extraire
3. **Fusionner des dataframes** avec des identifiants différents
4. **Gérer les versions de gènes** (ENSMUSG.version vs ENSMUSG)
5. **Sauvegarder** des données en format .RData et texte tabulé

<div class="alert alert-warning">
<b>Attention :</b> Ce travail sera évalué et compte pour votre note de contrôle continu. Suivez attentivement les instructions et n'hésitez pas à demander de l'aide si nécessaire.
</div>

## 1. Lecture des fichiers de données

Nous allons commencer par charger les trois fichiers nécessaires :
- Les comptages que vous avez générés (16 échantillons)
- Les comptages des auteurs (16 échantillons)  
- Les métadonnées décrivant les échantillons

### Chemins des fichiers de données

Les données sont stockées dans le répertoire `/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/4-counts/` :

- **Vos comptages** : `16samples_counts_matrix.txt`
- **Comptages auteurs** : `GSE76872_cortexGeneCounts_authors.csv`  
- **Métadonnées** : `SraRunTable-Blumenthal2014-cortex.tsv`

<div class="alert alert-info">
<b>Note technique :</b> Les auteurs ont utilisé TopHat2 sur le génome mm10 en 2013, tandis que vous avez utilisé STAR et featureCounts. On comparera les résultats des deux workflows plus loin dans le Fil Rouge.
</div>

In [ ]:
# Chargement des comptages étudiants (votre pipeline STAR + featureCounts)
student_counts <- read.table("/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/4-counts/16samples_counts_matrix.txt",
                            header = TRUE,
                            sep = "\t",
                            row.names = 1)

cat("Dimensions des comptages étudiants:", dim(student_counts), "\n")
cat("Colonnes:", colnames(student_counts), "\n")
head(student_counts, 3)

In [ ]:
# Chargement des comptages des auteurs (pipeline TopHat2 + BEDTools)
author_counts <- read.csv("/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/4-counts/GSE76872_cortexGeneCounts_authors.csv",
                         header = TRUE,
                         row.names = 4)  # La colonne 4 contient les IDs de gènes

author_counts <- author_counts[, -(1:3)]

cat("Dimensions des comptages auteurs:", dim(author_counts), "\n")
cat("Colonnes:", colnames(author_counts), "\n")
head(author_counts, 3)

In [ ]:
# Chargement des métadonnées
metadata <- read.table("/srv/data/meg-m1-a2b/blumenthal-2014/wholeData/4-counts/SraRunTable-Blumenthal2014-cortex.tsv",
                      header = TRUE,
                      sep = "\t")

metadata_key <- metadata[, c("Run", "animal_id", "gender", "genotype")]

cat("Dimensions des métadonnées:", dim(metadata_key), "\n")
print(metadata_key)

## 2. Analyse et nettoyage des identifiants

Avant de fusionner les données, nous devons résoudre plusieurs problèmes d'identifiants :

1. **Identifiants d'échantillons différents** :
   - Étudiants : SRX1589825, SRX1589826, etc.
   - Auteurs : T1979_ctx, T1980_ctx, etc.

2. **Versions de gènes différentes** :
   - Étudiants : ENSMUSG00000051951.6 (avec version)
   - Auteurs : ENSMUSG00000051951 (sans version)

In [ ]:
# Nettoyage des IDs de gènes étudiants (suppression des versions)

student_gene_ids <- rownames(student_counts)
cat("Exemples d'IDs étudiants avant nettoyage:\n")
print(head(student_gene_ids, 5))

clean_student_ids <- sub("\\..*", "", student_gene_ids)
rownames(student_counts) <- clean_student_ids

cat("\nExemples d'IDs étudiants après nettoyage:\n")
print(head(rownames(student_counts), 5))

cat("\nExemples d'IDs auteurs:\n")
print(head(rownames(author_counts), 5))

## 3. Création de la table de correspondance des échantillons

Nous devons créer une table qui fait le lien entre :
- Les identifiants SRX (colonnes des comptages étudiants)
- Les identifiants T#### (colonnes des comptages auteurs)
- Les informations de génotype et sexe

In [ ]:
# Extraction des identifiants SRX depuis les noms de colonnes étudiants
student_sample_ids <- sub("\\.full\\.bam", "", colnames(student_counts))
cat("Identifiants échantillons étudiants:\n")
print(student_sample_ids)

author_sample_ids <- sub("_ctx", "", colnames(author_counts))
cat("\nIdentifiants échantillons auteurs:\n")
print(author_sample_ids)

sample_mapping <- data.frame(
  SRX_id = student_sample_ids,
  T_id = author_sample_ids,
  stringsAsFactors = FALSE
)

sample_mapping <- merge(sample_mapping, metadata_key, 
                       by.x = "SRX_id", by.y = "Run", 
                       all.x = TRUE)

print("Table de correspondance des échantillons :")
print(sample_mapping)

## 4. Renommage des colonnes selon la nomenclature demandée

Nous devons renommer les colonnes selon le format :
**[génotype]_[sexe]_[numéro]_[source]**

Où :
- **génotype** : `wtdel`, `wtdup`, `del`, ou `dup`
- **sexe** : `f` (female) ou `m` (male)
- **numéro** : `1` ou `2` (première ou deuxième souris de cette condition)
- **source** : `s` (students) ou `a` (authors)

In [ ]:
# Fonction pour créer les nouveaux noms de colonnes
create_column_name <- function(genotype, gender, source) {
  if (genotype == "Wild Type") {
    geno_prefix <- "wt"  # On déterminera del/dup plus tard
  } else if (genotype == "Deletion") {
    geno_prefix <- "del"
  } else if (genotype == "Duplication") {
    geno_prefix <- "dup"
  }
  
  sex_code <- ifelse(gender == "female", "f", "m")
  
  return(paste(geno_prefix, sex_code, source, sep = "_"))
}

sample_mapping$student_name <- create_column_name(sample_mapping$genotype, 
                                                 sample_mapping$gender, "s")
sample_mapping$author_name <- create_column_name(sample_mapping$genotype, 
                                                sample_mapping$gender, "a")

print("Table avec nouveaux noms de colonnes :")
print(sample_mapping[, c("SRX_id", "T_id", "genotype", "gender", "student_name", "author_name")])

In [ ]:
# Ajout des numéros de réplicats (1 ou 2 pour chaque condition)

library(dplyr)

sample_mapping <- sample_mapping %>%
  group_by(genotype, gender) %>%
  mutate(replicate = row_number()) %>%
  ungroup()

sample_mapping$student_name_final <- paste0(
  sub("_s$", "", sample_mapping$student_name), 
  "_", sample_mapping$replicate, "_s"
)

sample_mapping$author_name_final <- paste0(
  sub("_a$", "", sample_mapping$author_name), 
  "_", sample_mapping$replicate, "_a"
)

print("Table finale avec numéros de réplicats :")
print(sample_mapping[, c("SRX_id", "T_id", "genotype", "gender", "student_name_final", "author_name_final")])

## 5. Application du renommage aux dataframes

Maintenant nous appliquons les nouveaux noms de colonnes aux deux dataframes de comptage.

In [ ]:
# Renommage des colonnes du dataframe étudiant
old_student_names <- colnames(student_counts)
new_student_names <- sample_mapping$student_name_final[match(sub("\\.full\\.bam", "", old_student_names), 
                                                            sample_mapping$SRX_id)]
colnames(student_counts) <- new_student_names

cat("Colonnes étudiants renommées :\n")
print(colnames(student_counts))

old_author_names <- colnames(author_counts)
new_author_names <- sample_mapping$author_name_final[match(sub("_ctx", "", old_author_names), 
                                                          sample_mapping$T_id)]
colnames(author_counts) <- new_author_names

cat("\nColonnes auteurs renommées :\n")
print(colnames(author_counts))

## 6. Fusion des dataframes

Nous fusionnons maintenant les deux dataframes en utilisant les identifiants de gènes comme clé de fusion. Attention : tous les gènes ne sont pas présents dans les deux jeux de données !

In [ ]:
# Vérification du chevauchement des gènes
student_genes <- rownames(student_counts)
author_genes <- rownames(author_counts)

common_genes <- intersect(student_genes, author_genes)
student_only <- setdiff(student_genes, author_genes)
author_only <- setdiff(author_genes, student_genes)

cat("Statistiques de chevauchement des gènes :\n")
cat("- Gènes étudiants :", length(student_genes), "\n")
cat("- Gènes auteurs :", length(author_genes), "\n")
cat("- Gènes communs :", length(common_genes), "\n")
cat("- Gènes uniquement étudiants :", length(student_only), "\n")
cat("- Gènes uniquement auteurs :", length(author_only), "\n")

In [ ]:
# Fusion des dataframes (outer join pour garder tous les gènes)
GUP <- merge(student_counts, author_counts, 
             by = "row.names", 
             all = TRUE)

rownames(GUP) <- GUP$Row.names
GUP$Row.names <- NULL

GUP[is.na(GUP)] <- 0

cat("Dimensions du dataframe fusionné GUP :", dim(GUP), "\n")
cat("Colonnes :", colnames(GUP), "\n")
cat("\nPremières lignes :\n")
print(head(GUP, 3))

## 7. Sauvegarde des résultats

Nous sauvegardons le dataframe final dans deux formats :
1. Format .RData pour une utilisation ultérieure en R
2. Format texte tabulé pour une utilisation dans d'autres logiciels

In [ ]:
# Sauvegarde en format .RData
save(GUP, file = "GUP_VotreNom1_VotreNom2.RData")

write.table(GUP, file = "GUP_VotreNom1_VotreNom2.txt", 
           sep = "\t", 
           quote = FALSE, 
           row.names = TRUE, 
           col.names = NA)

cat("Fichiers sauvegardés :\n")
cat("- GUP_VotreNom1_VotreNom2.RData\n")
cat("- GUP_VotreNom1_VotreNom2.txt\n")

cat("\nRésumé final du dataframe GUP :\n")
cat("- Dimensions :", dim(GUP), "\n")
cat("- Nombre de colonnes étudiants :", sum(grepl("_s$", colnames(GUP))), "\n")
cat("- Nombre de colonnes auteurs :", sum(grepl("_a$", colnames(GUP))), "\n")

## Conclusion et points clés


1. **Lecture de fichiers** avec différents formats (txt, csv, tsv)
2. **Nettoyage d'identifiants** (suppression de versions de gènes)
3. **Mapping d'identifiants** entre différents systèmes de nommage
4. **Renommage systématique** de colonnes selon une nomenclature précise
5. **Fusion de dataframes** avec gestion des données manquantes
6. **Sauvegarde** dans différents formats


- **Identifiants différents** : SRX vs T#### résolus via métadonnées
- **Versions de gènes** : ENSMUSG.version vs ENSMUSG harmonisés
- **Données manquantes** : gènes présents dans un seul jeu de données
- **Nomenclature complexe** : génotype + sexe + réplicat + source


Ce dataframe `GUP` sera utilisé dans la suite du Fil Rouge pour :
- Comparer les deux pipelines bioinformatiques
- Analyser l'expression différentielle
- Étudier l'impact des variations génomiques sur l'expression

<div class="alert alert-success">
<b>Félicitations !</b> Vous avez successfully fusionné des données de comptage RNASeq provenant de deux pipelines différents. Cette compétence est essentielle en bioinformatique pour intégrer des données hétérogènes.
</div>